Import Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sklearn
from scipy.stats import randint, loguniform, uniform

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.7.2


Load Dataset

In [2]:
try:
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = next(iter(uploaded))
except Exception:
    # For local Jupyter, place the CSV beside this notebook or edit this path.
    DATA_PATH = r"C:\Users\DHARVIN\Documents\Semester 7\Machine Learning and Related Applications\ecommerce_customer_churn_dataset.csv"

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("\\nTarget distribution:")
display(df["Churned"].value_counts(normalize=True).rename("proportion"))

Dataset shape: (50000, 25)


,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Email_Open_Rate,Customer_Service_Calls,Product_Reviews_Written,Social_Media_Engagement_Score,Mobile_App_Usage,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter
0,43.0,Male,France,Marseille,2.9,14.0,27.4,6.0,50.6,3.0,...,17.9,9.0,4.0,16.3,20.8,1.0,953.33,2278.0,0,Q1
1,36.0,Male,UK,Manchester,1.6,15.0,42.7,10.3,37.7,1.0,...,42.8,7.0,3.0,NaN,23.3,3.0,1067.47,3028.0,0,Q4
2,45.0,Female,Canada,Vancouver,2.9,10.0,24.8,1.6,70.9,1.0,...,0.0,4.0,1.0,NaN,8.8,NaN,1289.75,2317.0,0,Q4
3,56.0,Female,USA,New York,2.6,10.0,38.4,14.8,41.7,9.0,...,41.4,2.0,5.0,85.9,31.0,3.0,2340.92,2674.0,0,Q1
4,35.0,Male,India,Delhi,3.1,29.0,51.4,NaN,19.1,9.0,...,37.9,1.0,11.0,83.0,50.4,4.0,3041.29,5354.0,0,Q4


\nTarget distribution:


Churned
0    0.711
1    0.289
Name: proportion, dtype: float64

Prprocessing

In [3]:
TARGET = "Churned"
required = {
    "Age", "Gender", "Country", "City", "Membership_Years",
    "Login_Frequency", "Session_Duration_Avg", "Pages_Per_Session",
    "Cart_Abandonment_Rate", "Wishlist_Items", "Total_Purchases",
    "Average_Order_Value", "Days_Since_Last_Purchase", "Discount_Usage_Rate",
    "Returns_Rate", "Email_Open_Rate", "Customer_Service_Calls",
    "Product_Reviews_Written", "Social_Media_Engagement_Score",
    "Mobile_App_Usage", "Payment_Method_Diversity", "Lifetime_Value",
    "Credit_Balance", "Churned", "Signup_Quarter"
}
missing_columns = required.difference(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing expected columns: {sorted(missing_columns)}")

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].astype(int).copy()

if set(y.unique()) != {0, 1}:
    raise ValueError("Churned must contain binary labels 0 and 1.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

Training rows: 40000 | Test rows: 10000


In [4]:
class Chapter4CleaningAndFeatures(BaseEstimator, TransformerMixin):
    """Apply the report's value corrections and add its five engineered features."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Correct implausible values as described in Chapter 4.
        X.loc[(X["Age"] < 18) | (X["Age"] > 100), "Age"] = np.nan
        X.loc[X["Total_Purchases"] < 0, "Total_Purchases"] = np.nan
        X["Cart_Abandonment_Rate"] = X["Cart_Abandonment_Rate"].clip(upper=100)
        X["Discount_Usage_Rate"] = X["Discount_Usage_Rate"].clip(upper=100)
        X["Total_Purchases"] = X["Total_Purchases"].round()

        # Avoid division by zero; missing values are subsequently median-imputed.
        tenure = X["Membership_Years"].where(X["Membership_Years"] > 0, np.nan)
        X["Purchase_Frequency"] = X["Total_Purchases"] / tenure
        X["Estimated_Spend"] = X["Total_Purchases"] * X["Average_Order_Value"]
        X["Engagement_Index"] = (
            X["Login_Frequency"] *
            X["Session_Duration_Avg"] *
            X["Pages_Per_Session"]
        )
        X["Service_Calls_per_Year"] = X["Customer_Service_Calls"] / tenure
        X["Recency_to_Tenure"] = (
            X["Days_Since_Last_Purchase"] / (tenure * 365)
        )

        # City was dropped in Chapter 4; the remaining categorical columns are encoded.
        return X.drop(columns=["City"])


class FoldWinsorizer(BaseEstimator, TransformerMixin):
    """Learn 1st/99th percentile caps on each training fold only."""
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lower_bounds_ = np.nanquantile(X, self.lower, axis=0)
        self.upper_bounds_ = np.nanquantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.clip(X, self.lower_bounds_, self.upper_bounds_)


numeric_features = [
    "Age", "Membership_Years", "Login_Frequency", "Session_Duration_Avg",
    "Pages_Per_Session", "Cart_Abandonment_Rate", "Wishlist_Items",
    "Total_Purchases", "Average_Order_Value", "Days_Since_Last_Purchase",
    "Discount_Usage_Rate", "Returns_Rate", "Email_Open_Rate",
    "Customer_Service_Calls", "Product_Reviews_Written",
    "Social_Media_Engagement_Score", "Mobile_App_Usage",
    "Payment_Method_Diversity", "Lifetime_Value", "Credit_Balance",
    "Purchase_Frequency", "Estimated_Spend", "Engagement_Index",
    "Service_Calls_per_Year", "Recency_to_Tenure"
]
categorical_features = ["Gender", "Country", "Signup_Quarter"]

numeric_pipeline = Pipeline(steps=[
    ("winsor", FoldWinsorizer(0.01, 0.99)),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

def make_pipeline(classifier):
    return Pipeline(steps=[
        ("features", Chapter4CleaningAndFeatures()),
        ("preprocess", preprocessor),
        ("classifier", classifier)
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = "roc_auc"

Baseline Models and Search 

In [5]:
models = {
    "Logistic Regression": make_pipeline(
        LogisticRegression(
            C=1.0, penalty="l2", solver="lbfgs",
            max_iter=3000, class_weight="balanced", random_state=42
        )
    ),
    "Random Forest": make_pipeline(
        RandomForestClassifier(
            n_estimators=100, class_weight="balanced_subsample",
            random_state=42, n_jobs=1
        )
    ),
    "Histogram Gradient Boosting": make_pipeline(
        HistGradientBoostingClassifier(
            max_iter=100, learning_rate=0.1, max_leaf_nodes=31,
            class_weight="balanced", early_stopping=False,
            random_state=42
        )
    )
}

grid_spaces = {
    "Logistic Regression": {
        "classifier__C": [0.1, 1.0, 10.0]
    },
    "Random Forest": {
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [None, 15],
        "classifier__min_samples_split": [2, 10],
        "classifier__min_samples_leaf": [1, 3]
    },
    "Histogram Gradient Boosting": {
        "classifier__learning_rate": [0.05, 0.1],
        "classifier__max_iter": [100, 200],
        "classifier__max_leaf_nodes": [15, 31],
        "classifier__l2_regularization": [0.0, 1.0]
    }
}

random_spaces = {
    "Logistic Regression": {
        "classifier__C": loguniform(0.01, 100)
    },
    "Random Forest": {
        "classifier__n_estimators": randint(100, 301),
        "classifier__max_depth": [None, 10, 15, 20, 30],
        "classifier__min_samples_split": randint(2, 16),
        "classifier__min_samples_leaf": randint(1, 8)
    },
    "Histogram Gradient Boosting": {
        "classifier__learning_rate": loguniform(0.02, 0.2),
        "classifier__max_iter": randint(75, 251),
        "classifier__max_leaf_nodes": randint(10, 41),
        "classifier__l2_regularization": uniform(0.0, 3.0)
    }
}

Run Grid Search

In [6]:
grid_results = {}
grid_rows = []

for name, pipe in models.items():
    print(f"Grid Search: {name}")
    search = GridSearchCV(
        estimator=pipe,
        param_grid=grid_spaces[name],
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=1,
        return_train_score=True
    )
    search.fit(X_train, y_train)
    grid_results[name] = search
    grid_rows.append({
        "Model": name,
        "Best CV ROC-AUC": search.best_score_,
        "Best parameters": search.best_params_
    })

grid_summary = pd.DataFrame(grid_rows).sort_values(
    "Best CV ROC-AUC", ascending=False
)
display(grid_summary)

Grid Search: Logistic Regression
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Grid Search: Random Forest
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Grid Search: Histogram Gradient Boosting
Fitting 5 folds for each of 16 candidates, totalling 80 fits


,Model,Best CV ROC-AUC,Best parameters
2,Histogram Gradient Boosting,0.926502,"{'classifier__l2_regularization': 1.0, 'classi..."
1,Random Forest,0.921672,"{'classifier__max_depth': None, 'classifier__m..."
0,Logistic Regression,0.798277,{'classifier__C': 0.1}


Run Random Search

In [7]:
N_ITER = 10
random_results = {}
random_rows = []

for name, pipe in models.items():
    print(f"Random Search: {name}")
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=random_spaces[name],
        n_iter=N_ITER,
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        refit=True,
        random_state=42,
        verbose=1,
        return_train_score=True
    )
    search.fit(X_train, y_train)
    random_results[name] = search
    random_rows.append({
        "Model": name,
        "Best CV ROC-AUC": search.best_score_,
        "Best parameters": search.best_params_
    })

random_summary = pd.DataFrame(random_rows).sort_values(
    "Best CV ROC-AUC", ascending=False
)
display(random_summary)

Random Search: Logistic Regression
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Random Search: Random Forest
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Random Search: Histogram Gradient Boosting
Fitting 5 folds for each of 10 candidates, totalling 50 fits


,Model,Best CV ROC-AUC,Best parameters
2,Histogram Gradient Boosting,0.925891,{'classifier__l2_regularization': 2.5985284373...
1,Random Forest,0.921346,"{'classifier__max_depth': 30, 'classifier__min..."
0,Logistic Regression,0.798279,{'classifier__C': 0.04207988669606638}


Compare Search Methods and Select the tuned Pipeline

In [8]:
selected_searches = {}
comparison_rows = []

for name in models:
    g = grid_results[name]
    r = random_results[name]
    if g.best_score_ >= r.best_score_:
        selected_searches[name] = g
        method = "Grid Search"
        best = g
    else:
        selected_searches[name] = r
        method = "Random Search"
        best = r

    comparison_rows.append({
        "Model": name,
        "Grid CV ROC-AUC": g.best_score_,
        "Random CV ROC-AUC": r.best_score_,
        "Selected method": method,
        "Selected CV ROC-AUC": best.best_score_,
        "Selected parameters": best.best_params_
    })

comparison = pd.DataFrame(comparison_rows).sort_values(
    "Selected CV ROC-AUC", ascending=False
)
display(comparison)

,Model,Grid CV ROC-AUC,Random CV ROC-AUC,Selected method,Selected CV ROC-AUC,Selected parameters
2,Histogram Gradient Boosting,0.926502,0.925891,Grid Search,0.926502,"{'classifier__l2_regularization': 1.0, 'classi..."
1,Random Forest,0.921672,0.921346,Grid Search,0.921672,"{'classifier__max_depth': None, 'classifier__m..."
0,Logistic Regression,0.798277,0.798279,Random Search,0.798279,{'classifier__C': 0.04207988669606638}


Final Test-Set Evaluation

In [9]:
test_rows = []
best_estimators = {}

for name, search in selected_searches.items():
    best_model = search.best_estimator_
    best_estimators[name] = best_model

    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]

    test_rows.append({
        "Model": name,
        "Test ROC-AUC": roc_auc_score(y_test, y_prob),
        "Test F1 (churn=1)": f1_score(y_test, y_pred, pos_label=1),
        "Test Precision (churn=1)": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Test Recall (churn=1)": recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    })

test_results = pd.DataFrame(test_rows).sort_values(
    "Test ROC-AUC", ascending=False
)
display(test_results.round(4))

for name, model in best_estimators.items():
    print("\\n", name)
    print(confusion_matrix(y_test, model.predict(X_test)))
    print(classification_report(y_test, model.predict(X_test), digits=3))

,Model,Test ROC-AUC,Test F1 (churn=1),Test Precision (churn=1),Test Recall (churn=1)
2,Histogram Gradient Boosting,0.9284,0.8403,0.8213,0.8602
1,Random Forest,0.9248,0.8445,0.8744,0.8166
0,Logistic Regression,0.8024,0.6090,0.5182,0.7384


\n Logistic Regression
[[5126 1984]
 [ 756 2134]]
              precision    recall  f1-score   support

           0      0.871     0.721     0.789      7110
           1      0.518     0.738     0.609      2890

    accuracy                          0.726     10000
   macro avg      0.695     0.730     0.699     10000
weighted avg      0.769     0.726     0.737     10000

\n Random Forest
[[6771  339]
 [ 530 2360]]
              precision    recall  f1-score   support

           0      0.927     0.952     0.940      7110
           1      0.874     0.817     0.845      2890

    accuracy                          0.913     10000
   macro avg      0.901     0.884     0.892     10000
weighted avg      0.912     0.913     0.912     10000

\n Histogram Gradient Boosting
[[6569  541]
 [ 404 2486]]
              precision    recall  f1-score   support

           0      0.942     0.924     0.933      7110
           1      0.821     0.860     0.840      2890

    accuracy                  